In [ ]:
# NN forward desing ,,,,, R**2 vs number of data#############################

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

# -------------------------------
# 1️⃣ Load dataset
# -------------------------------
file_path = '/content/sample_data/data 1.xlsx'
data = pd.read_excel(file_path)
data.columns = data.columns.str.strip().str.lower().str.replace(' ', '_')

# Inputs: thicknesses
X = data[['t1','t2','t3']].values.astype(np.float32)
# Target: total energy
y = data[['e_total']].values.astype(np.float32)

# -------------------------------
# 2️⃣ Normalize
# -------------------------------
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y)

X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_scaled, dtype=torch.float32)

# -------------------------------
# 3️⃣ Define Forward NN (MLP)
# -------------------------------
class ForwardMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x)

# -------------------------------
# 4️⃣ R² vs data numbers
# -------------------------------
subset_sizes = np.linspace(10, len(X), 50, dtype=int)  # 20 steps from 10 → full dataset
r2_values = []

for n_samples in subset_sizes:
    # Use first n_samples points
    X_sub = X_tensor[:n_samples]
    y_sub = y_tensor[:n_samples]

    model = ForwardMLP()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    # Train briefly (e.g., 1000 epochs)
    for epoch in range(1000):
        optimizer.zero_grad()
        y_pred = model(X_sub)
        loss = loss_fn(y_pred, y_sub)
        loss.backward()
        optimizer.step()

    # Predict and compute R²
    with torch.no_grad():
        y_pred_scaled = model(X_sub).numpy()
    y_pred_real = scaler_y.inverse_transform(y_pred_scaled)
    y_true_real = scaler_y.inverse_transform(y_sub.numpy())

    r2 = r2_score(y_true_real, y_pred_real)
    r2_values.append(r2)

# -------------------------------
# 5️⃣ Plot
# -------------------------------
plt.figure(figsize=(7,5))
plt.plot(subset_sizes, r2_values, marker='o')
plt.xlabel("Number of Training Samples")
plt.ylabel("R²")
plt.title("R² vs Number of Training Samples (Forward NN)")
plt.grid(True)
plt.show()

In [ ]:
# NN inverse desing ,,,,, R**2 vs number of data#############################

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

# -------------------------------
# 1️⃣ Load dataset
# -------------------------------
file_path = '/content/sample_data/data 2.xlsx'
data = pd.read_excel(file_path)
data.columns = data.columns.str.strip().str.lower().str.replace(' ', '_')

# Inputs: layer energies
X = data[['e1','e2','e3']].values.astype(np.float32)

# Targets: layer thicknesses
y = data[['t1','t2','t3']].values.astype(np.float32)

# Optional check (sanity)
data['e_total_check'] = data['e1'] + data['e2'] + data['e3']

# -------------------------------
# 2️⃣ Normalize
# -------------------------------
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y)

X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_scaled, dtype=torch.float32)

# -------------------------------
# 3️⃣ Define Inverse Model
# -------------------------------
class InverseLayerMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 64),   # input: e1, e2, e3
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 3)    # output: t1, t2, t3
        )
    def forward(self, x):
        return self.net(x)

# -------------------------------
# 4️⃣ R² vs number of samples
# -------------------------------
max_samples = min(10000, len(X))  # ensures no crash if dataset < 2000
subset_sizes = np.linspace(10, max_samples, 50, dtype=int)
r2_values = []

for n_samples in subset_sizes:

    X_sub = X_tensor[:n_samples]
    y_sub = y_tensor[:n_samples]

    model = InverseLayerMLP()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    # Training
    for epoch in range(1000):
        optimizer.zero_grad()
        y_pred = model(X_sub)
        loss = loss_fn(y_pred, y_sub)
        loss.backward()
        optimizer.step()

    # Prediction
    with torch.no_grad():
        y_pred_scaled = model(X_sub).numpy()

    y_pred_real = scaler_y.inverse_transform(y_pred_scaled)
    y_true_real = scaler_y.inverse_transform(y_sub.numpy())

    # R² per layer
    r2_t1 = r2_score(y_true_real[:,0], y_pred_real[:,0])
    r2_t2 = r2_score(y_true_real[:,1], y_pred_real[:,1])
    r2_t3 = r2_score(y_true_real[:,2], y_pred_real[:,2])

    # Mean R²
    r2_mean = (r2_t1 + r2_t2 + r2_t3) / 3
    r2_values.append(r2_mean)



# -------------------------------
# Find when R² approaches 1
# -------------------------------
delta_r2 = np.diff(r2_values)

r2_threshold = 0.99
smooth_threshold = 0.01
window = 5  # consecutive stable points

for i in range(len(delta_r2) - window):

    r2_condition = r2_values[i] >= r2_threshold
    smooth_condition = np.all(np.abs(delta_r2[i:i+window]) < smooth_threshold)

    if r2_condition and smooth_condition:
        print(f"Converged (R²≈1 and smooth) at ~{subset_sizes[i]} samples")
        break


# -------------------------------
# 5️⃣ Plot
# -------------------------------
plt.figure(figsize=(7,5))
plt.plot(subset_sizes, r2_values, marker='o')
plt.xlabel("Number of Training Samples")
plt.ylabel("R² (Layer-wise Inverse Design)")
plt.title("Inverse Design (e1,e2,e3 → t1,t2,t3)")
plt.grid(True)
plt.show()




plt.figure(figsize=(7,5))
plt.plot(subset_sizes[1:], delta_r2, marker='o')
plt.axhline(y=0, linestyle='--')
plt.xlabel("Number of Samples")
plt.ylabel("ΔR² (change)")
plt.title("Smoothness of Learning Curve")
plt.grid(True)
plt.show()

In [ ]:
# =========================================
# Simple Neural Network (Forward Design)------figure 8
# =========================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# -------------------------------
# 1️⃣ Load data
# -------------------------------
file_path = '/content/sample_data/data_new.xlsx'
data = pd.read_excel(file_path)

data.columns = data.columns.str.strip().str.lower().str.replace(' ', '_')
dataset = data[['e_total', 'm_total']].dropna()

# -------------------------------
# 🎨 Style
# -------------------------------
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "figure.dpi": 300
})

# -------------------------------
# 2️⃣ All points
# -------------------------------
plt.scatter(dataset['m_total'], dataset['e_total'],
            color='#b5b5b5', s=8, alpha=0.25,
            edgecolors='none', label='All designs')

# -------------------------------
# 3️⃣ Pareto frontier (upper envelope)
# -------------------------------
# sort by mass (x-axis)
sorted_data = dataset.sort_values(by='m_total')

pareto = []
max_energy = -np.inf

for _, row in sorted_data.iterrows():
    if row['e_total'] > max_energy:
        pareto.append(row)
        max_energy = row['e_total']

pareto_df = pd.DataFrame(pareto)

# -------------------------------
# 4️⃣ Plot Pareto curve (RED LINE)
# -------------------------------
plt.plot(pareto_df['m_total'],
         pareto_df['e_total'],
         color='red',
         linewidth=2.5,
         label='Pareto Front')

plt.scatter(pareto_df['m_total'],
            pareto_df['e_total'],
            color='red',
            s=25)

# -------------------------------
# 5️⃣ Labels
# -------------------------------
plt.xlabel("Total Mass")
plt.ylabel("Total Energy")
plt.title("Pareto Optimal Boundary (Mass vs Energy)")
plt.legend()

plt.tight_layout()
plt.savefig("pareto_boundary.pdf")
plt.show()


# -------------------------------
# 📊 3️⃣ Scatter plots (t vs E_total)
# -------------------------------
def plot_t_vs_E(t, color):
    plt.figure(figsize=(6,5))
    sns.scatterplot(x=dataset[t], y=dataset['e_total'],
                    color=color, s=40, edgecolor='black')

    plt.xlabel(f"{t}")
    plt.ylabel("E_total")
    plt.title(f"{t} vs E_total")

    plt.tight_layout()
    plt.savefig(f"{t}_vs_E_total.pdf")
    plt.close()

plot_t_vs_E('t1', '#1f77b4')  # blue
plot_t_vs_E('t2', '#2ca02c')  # green
plot_t_vs_E('t3', '#d62728')  # red

# -------------------------------
# 📈 4️⃣ Histograms
# -------------------------------
for col in dataset.columns:
    plt.figure(figsize=(6,5))
    sns.histplot(dataset[col], kde=True, color='#4c72b0')

    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.title(f"Distribution of {col}")

    plt.tight_layout()
    plt.savefig(f"hist_{col}.pdf")
    plt.close()

# -------------------------------
# 🔥 5️⃣ Correlation heatmap
# -------------------------------
plt.figure(figsize=(8,6))

corr = dataset.corr()

sns.heatmap(corr, annot=True, cmap="coolwarm",
            fmt=".2f", linewidths=0.5)

plt.title("Correlation Matrix")
plt.tight_layout()
plt.savefig("correlation_matrix.pdf")
plt.close()

# -------------------------------
# 📉 6️⃣ Pair plot (high quality)
# -------------------------------
sns.pairplot(dataset,
             diag_kind='kde',
             plot_kws={'alpha':0.6, 's':30, 'edgecolor':'black'})

plt.savefig("pairplot.pdf")
plt.close()

# -------------------------------
# 📊 Combined scatter plot
# -------------------------------
plt.figure(figsize=(6,5))

# Plot each variable with different color
plt.scatter(dataset['t1'], dataset['e_total'],
            color='#1f77b4', label='t1', s=40)

plt.scatter(dataset['t2'], dataset['e_total'],
            color='#2ca02c', label='t2', s=40)

plt.scatter(dataset['t3'], dataset['e_total'],
            color='#d62728', label='t3', s=40)

# Labels & styling
plt.xlabel("t (mm)")
plt.ylabel("E_total")
plt.title("t1, t2, t3 vs E_total")

plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("t_all_vs_E_total.pdf")
plt.show()

In [ ]:
# NN inverse design #####################################

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler

# -------------------------------
# 1️⃣ Load dataset
# -------------------------------
file_path = '/content/sample_data/data_new.xlsx'
data = pd.read_excel(file_path)

data.columns = data.columns.str.strip().str.lower().str.replace(' ', '_')

dataset = data[['t1','t2','t3','e_total','m_total']]

# -------------------------------
# 2️⃣ Prepare inverse design data
# -------------------------------
# INPUT: mass + energy
X = dataset[['m_total', 'e_total']].values.astype(np.float32)

# OUTPUT: geometry
y = dataset[['t1', 't2', 't3']].values.astype(np.float32)

# -------------------------------
# 3️⃣ Normalize data
# -------------------------------
x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X_scaled = x_scaler.fit_transform(X)
y_scaled = y_scaler.fit_transform(y)

# Convert to tensors
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_scaled, dtype=torch.float32)

# -------------------------------
# 4️⃣ Define Inverse Neural Network
# -------------------------------
class InverseNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 3)
        )

    def forward(self, x):
        return self.net(x)

model = InverseNN()

# -------------------------------
# 5️⃣ Training setup
# -------------------------------
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# -------------------------------
# 6️⃣ Training loop
# -------------------------------
epochs = 2000

for epoch in range(epochs):
    pred = model(X_tensor)
    loss = criterion(pred, y_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 200 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.6f}")

# -------------------------------
# 7️⃣ Inverse design prediction
# -------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------------
# Force vector-quality PDF output
# -------------------------------
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

sns.set_theme(style="whitegrid")

fig = plt.figure(figsize=(7,6))

# -------------------------------
# t1
# -------------------------------
plt.scatter(dataset['e_total'], dataset['t1'],
            color='#1f77b4', s=40, alpha=0.7,
            edgecolors='black', linewidth=0.3,
            label='$t_1$')

# -------------------------------
# t2
# -------------------------------
plt.scatter(dataset['e_total'], dataset['t2'],
            color='#2ca02c', s=40, alpha=0.7,
            edgecolors='black', linewidth=0.3,
            label='$t_2$')

# -------------------------------
# t3
# -------------------------------
plt.scatter(dataset['e_total'], dataset['t3'],
            color='#d62728', s=40, alpha=0.7,
            edgecolors='black', linewidth=0.3,
            label='$t_3$')

# -------------------------------
# Labels & style
# -------------------------------
plt.xlabel("Total Energy, $E_{total}$", fontsize=13)
plt.ylabel("Thickness Parameters (t1, t2, t3)", fontsize=13)
plt.title("Inverse Design: Geometry vs Energy", fontsize=14)

plt.grid(True, linestyle='--', alpha=0.4)
plt.legend()

plt.tight_layout()

# -------------------------------
# SAVE AS HIGH-QUALITY PDF
# -------------------------------
plt.savefig(
    "inverse_design_high_quality.pdf",
    format="pdf",
    bbox_inches="tight"
)

plt.show()


######## mtotal vs t1 t2 t3

import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------------
# Style + PDF quality settings
# -------------------------------
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

sns.set_theme(style="whitegrid")

plt.figure(figsize=(7,6))

# -------------------------------
# t1 vs mass
# -------------------------------
plt.scatter(dataset['t1'], dataset['m_total'],
            color='#1f77b4', s=40, alpha=0.7,
            edgecolors='black', linewidth=0.3,
            label='$t_1$')

# -------------------------------
# t2 vs mass
# -------------------------------
plt.scatter(dataset['t2'], dataset['m_total'],
            color='#2ca02c', s=40, alpha=0.7,
            edgecolors='black', linewidth=0.3,
            label='$t_2$')

# -------------------------------
# t3 vs mass
# -------------------------------
plt.scatter(dataset['t3'], dataset['m_total'],
            color='#d62728', s=40, alpha=0.7,
            edgecolors='black', linewidth=0.3,
            label='$t_3$')

# -------------------------------
# Labels
# -------------------------------
plt.xlabel("Thickness Parameters (t1, t2, t3)", fontsize=13)
plt.ylabel("Total Mass $m_{total}$", fontsize=13)
plt.title("Forward Design: Geometry vs Mass", fontsize=14)

plt.grid(True, linestyle='--', alpha=0.4)
plt.legend()

plt.tight_layout()

# -------------------------------
# Save high-quality PDF
# -------------------------------
plt.savefig("mass_vs_t_all.pdf", format="pdf", bbox_inches="tight")

plt.show()



# -------------------------------
# 7️⃣ Min-mass selection per energy
# -------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

# Avoid modifying original data
dataset_plot = dataset.copy()

# -------------------------------
# Create energy bins
# -------------------------------
dataset_plot['energy_bin'] = pd.cut(dataset_plot['e_total'], bins=30)

# -------------------------------
# Select min mass in each bin
# -------------------------------
min_mass_designs = dataset_plot.loc[
    dataset_plot.groupby('energy_bin')['m_total'].idxmin()
]

# Sort for nicer plotting
min_mass_designs = min_mass_designs.sort_values(by='e_total')

# -------------------------------
# Plot
# -------------------------------
sns.set_theme(style="whitegrid")

plt.figure(figsize=(7,6))

# t1
plt.plot(min_mass_designs['e_total'], min_mass_designs['t1'],
         marker='o', color='#1f77b4', label='$t_1$')

# t2
plt.plot(min_mass_designs['e_total'], min_mass_designs['t2'],
         marker='o', color='#2ca02c', label='$t_2$')

# t3
plt.plot(min_mass_designs['e_total'], min_mass_designs['t3'],
         marker='o', color='#d62728', label='$t_3$')

# -------------------------------
# Labels
# -------------------------------
plt.xlabel("Total Energy $E_{total}$", fontsize=13)
plt.ylabel("Thickness Parameters", fontsize=13)
plt.title("Minimum-Mass Geometry for Each Energy Level", fontsize=14)

plt.grid(True, linestyle='--', alpha=0.4)
plt.legend()

plt.tight_layout()

# -------------------------------
# Save
# -------------------------------
plt.savefig("min_mass_per_energy.pdf", format="pdf", bbox_inches="tight")

plt.show()